In [1]:
from google.colab import files
uploaded = files.upload()

Saving Walmartsales Dataset (1).xlsx to Walmartsales Dataset (1).xlsx


In [3]:
import pandas as pd
df = pd.read_excel('Walmartsales Dataset (1).xlsx')
df.head()

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating,Customer ID
0,750-67-8428,A,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,2019-05-01 00:00:00,13:08:00,Ewallet,522.83,4.761905,26.1415,9.1,2
1,226-31-3081,C,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,2019-08-03 00:00:00,10:29:00,Cash,76.40,4.761905,3.8200,9.6,3
2,631-41-3108,A,Yangon,Normal,Male,Home and lifestyle,46.33,7,16.2155,340.5255,2019-03-03 00:00:00,13:23:00,Credit card,324.31,4.761905,16.2155,7.4,11
3,123-19-1176,A,Yangon,Member,Male,Health and beauty,58.22,8,23.2880,489.0480,27-01-2019,20:33:00,Ewallet,465.76,4.761905,23.2880,8.4,11
4,373-73-7910,A,Yangon,Normal,Male,Sports and travel,86.31,7,30.2085,634.3785,2019-08-02 00:00:00,10:37:00,Ewallet,604.17,4.761905,30.2085,5.3,9


In [4]:
import sqlite3

# Create a SQLite database
conn = sqlite3.connect('walmart.db')

# Save the Excel data as a SQL table
df.to_sql('walmart_sales', conn, if_exists='replace', index=False)

print("Data imported successfully into SQL table: walmart_sales")

Data imported successfully into SQL table: walmart_sales


In [5]:
query = "SELECT * FROM walmart_sales LIMIT 5;"
result = pd.read_sql_query(query, conn)
result

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating,Customer ID
0,750-67-8428,A,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,2019-05-01 00:00:00,13:08:00.000000,Ewallet,522.83,4.761905,26.1415,9.1,2
1,226-31-3081,C,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,2019-08-03 00:00:00,10:29:00.000000,Cash,76.40,4.761905,3.8200,9.6,3
2,631-41-3108,A,Yangon,Normal,Male,Home and lifestyle,46.33,7,16.2155,340.5255,2019-03-03 00:00:00,13:23:00.000000,Credit card,324.31,4.761905,16.2155,7.4,11
3,123-19-1176,A,Yangon,Member,Male,Health and beauty,58.22,8,23.2880,489.0480,27-01-2019,20:33:00.000000,Ewallet,465.76,4.761905,23.2880,8.4,11
4,373-73-7910,A,Yangon,Normal,Male,Sports and travel,86.31,7,30.2085,634.3785,2019-08-02 00:00:00,10:37:00.000000,Ewallet,604.17,4.761905,30.2085,5.3,9


In [6]:
query = """
SELECT branch, ROUND(SUM(total), 2) AS total_sales
FROM walmart_sales
GROUP BY branch
ORDER BY total_sales DESC;
"""

result = pd.read_sql_query(query, conn)
result

,Branch,total_sales
0,C,110568.71
1,A,106200.37
2,B,106197.67


In [8]:
query = """
WITH monthly_sales AS (
    SELECT
        branch,
        strftime('%Y-%m', date) AS month,
        SUM(total) AS total_sales
    FROM walmart_sales
    GROUP BY branch, month
),
sales_growth AS (
    SELECT
        branch,
        month,
        total_sales,
        LAG(total_sales) OVER (
            PARTITION BY branch
            ORDER BY month
        ) AS previous_month_sales
    FROM monthly_sales
)
SELECT
    branch,
    ROUND(
        AVG(
            CASE
                WHEN previous_month_sales IS NOT NULL
                     AND previous_month_sales <> 0
                THEN ((total_sales - previous_month_sales) / previous_month_sales) * 100
            END
        ),
        2
    ) AS avg_growth_rate
FROM sales_growth
GROUP BY branch
ORDER BY avg_growth_rate DESC;
"""

result = pd.read_sql_query(query, conn)
result

,branch,avg_growth_rate
0,B,17.98
1,C,5.96
2,A,-3.28


In [10]:
query = """
WITH product_profit AS (
    SELECT
        Branch AS branch,
        "Product line" AS product_line,
        SUM("Gross income" - cogs) AS profit
    FROM walmart_sales
    GROUP BY Branch, "Product line"
),
ranked_profit AS (
    SELECT
        branch,
        product_line,
        ROUND(profit, 2) AS profit,
        ROW_NUMBER() OVER (
            PARTITION BY branch
            ORDER BY profit DESC
        ) AS rn
    FROM product_profit
)
SELECT
    branch,
    product_line,
    profit
FROM ranked_profit
WHERE rn = 1
ORDER BY branch;
"""

result = pd.read_sql_query(query, conn)
result

,branch,product_line,profit
0,A,Health and beauty,-11397.97
1,B,Food and beverages,-13765.85
2,C,Home and lifestyle,-12572.17


In [12]:
query = """
WITH customer_spending AS (
    SELECT
        "Customer type" AS customer_type,
        ROUND(AVG(Total), 2) AS avg_spending
    FROM walmart_sales
    GROUP BY "Customer type"
)
SELECT
    customer_type,
    avg_spending,
    CASE
        WHEN avg_spending >= 350 THEN 'High'
        WHEN avg_spending >= 200 THEN 'Medium'
        ELSE 'Low'
    END AS spending_tier
FROM customer_spending
ORDER BY avg_spending DESC;
"""

result = pd.read_sql_query(query, conn)
result

,customer_type,avg_spending,spending_tier
0,Member,327.79,Medium
1,Normal,318.12,Medium


In [13]:
query = """
WITH product_avg AS (
    SELECT
        "Product line" AS product_line,
        AVG(Total) AS avg_total
    FROM walmart_sales
    GROUP BY "Product line"
)
SELECT
    w."Invoice ID" AS invoice_id,
    w."Product line" AS product_line,
    ROUND(w.Total, 2) AS transaction_total,
    ROUND(p.avg_total, 2) AS average_product_total,
    CASE
        WHEN w.Total > p.avg_total * 1.5 THEN 'High Anomaly'
        WHEN w.Total < p.avg_total * 0.5 THEN 'Low Anomaly'
        ELSE 'Normal'
    END AS anomaly_status
FROM walmart_sales w
JOIN product_avg p
    ON w."Product line" = p.product_line
WHERE w.Total > p.avg_total * 1.5
   OR w.Total < p.avg_total * 0.5
ORDER BY transaction_total DESC;
"""

result = pd.read_sql_query(query, conn)
result

,invoice_id,product_line,transaction_total,average_product_total,anomaly_status
0,860-79-0874,Fashion accessories,1042.65,305.09,High Anomaly
1,687-47-8271,Fashion accessories,1039.29,305.09,High Anomaly
2,283-26-5248,Food and beverages,1034.46,322.67,High Anomaly
3,751-41-9720,Home and lifestyle,1023.75,336.64,High Anomaly
4,303-96-2227,Home and lifestyle,1022.49,336.64,High Anomaly
...,...,...,...,...,...
564,236-86-3015,Home and lifestyle,14.68,336.64,Low Anomaly
565,192-98-7397,Fashion accessories,13.42,305.09,Low Anomaly
566,279-62-1445,Fashion accessories,13.17,305.09,Low Anomaly
567,308-39-1707,Fashion accessories,12.69,305.09,Low Anomaly


In [14]:
query = """
WITH payment_counts AS (
    SELECT
        City AS city,
        Payment AS payment_method,
        COUNT(*) AS usage_count
    FROM walmart_sales
    GROUP BY City, Payment
),
ranked_payments AS (
    SELECT
        city,
        payment_method,
        usage_count,
        ROW_NUMBER() OVER (
            PARTITION BY city
            ORDER BY usage_count DESC
        ) AS rn
    FROM payment_counts
)
SELECT
    city,
    payment_method,
    usage_count
FROM ranked_payments
WHERE rn = 1
ORDER BY city;
"""

result = pd.read_sql_query(query, conn)
result

,city,payment_method,usage_count
0,Mandalay,Ewallet,113
1,Naypyitaw,Cash,124
2,Yangon,Ewallet,126


In [17]:
query = """
SELECT
    strftime('%Y-%m', Date) AS month,
    Gender AS gender,
    ROUND(SUM(Total), 2) AS total_sales
FROM walmart_sales
GROUP BY month, Gender
ORDER BY month, Gender;
"""

result = pd.read_sql_query(query, conn)
result

,month,gender,total_sales
0,None,Female,93189.74
1,None,Male,94421.75
2,2019-01,Female,6397.93
3,2019-01,Male,3426.15
4,2019-02,Female,7477.21
5,2019-02,Male,5169.55
6,2019-03,Female,5793.65
7,2019-03,Male,6605.58
8,2019-04,Female,5491.00
9,2019-04,Male,2466.63


In [18]:
query = """
WITH sales_by_type AS (
    SELECT
        "Customer type" AS customer_type,
        "Product line" AS product_line,
        SUM(Total) AS total_sales
    FROM walmart_sales
    GROUP BY "Customer type", "Product line"
),
ranked AS (
    SELECT
        customer_type,
        product_line,
        ROUND(total_sales, 2) AS total_sales,
        ROW_NUMBER() OVER (
            PARTITION BY customer_type
            ORDER BY total_sales DESC
        ) AS rn
    FROM sales_by_type
)
SELECT
    customer_type,
    product_line,
    total_sales
FROM ranked
WHERE rn = 1
ORDER BY customer_type;
"""

result = pd.read_sql_query(query, conn)
result

,customer_type,product_line,total_sales
0,Member,Food and beverages,31357.62
1,Normal,Electronic accessories,29839.04


In [19]:
query = """
WITH purchases AS (
    SELECT
        "Customer type" AS customer_type,
        Date,
        LAG(Date) OVER (
            PARTITION BY "Customer type"
            ORDER BY Date
        ) AS prev_date
    FROM walmart_sales
)
SELECT
    customer_type,
    COUNT(*) AS repeat_purchases
FROM purchases
WHERE prev_date IS NOT NULL
  AND julianday(Date) - julianday(prev_date) <= 30
GROUP BY customer_type
ORDER BY repeat_purchases DESC;
"""

result = pd.read_sql_query(query, conn)
result

,customer_type,repeat_purchases
0,Member,215
1,Normal,196


In [20]:
query = """
SELECT
    "Customer type" AS customer_type,
    ROUND(SUM(Total), 2) AS total_sales
FROM walmart_sales
GROUP BY "Customer type"
ORDER BY total_sales DESC
LIMIT 5;
"""

result = pd.read_sql_query(query, conn)
result

,customer_type,total_sales
0,Member,164223.44
1,Normal,158743.31


In [21]:
query = """
SELECT
    CASE strftime('%w', Date)
        WHEN '0' THEN 'Sunday'
        WHEN '1' THEN 'Monday'
        WHEN '2' THEN 'Tuesday'
        WHEN '3' THEN 'Wednesday'
        WHEN '4' THEN 'Thursday'
        WHEN '5' THEN 'Friday'
        WHEN '6' THEN 'Saturday'
    END AS day_of_week,
    ROUND(SUM(Total), 2) AS total_sales
FROM walmart_sales
GROUP BY strftime('%w', Date)
ORDER BY total_sales DESC;
"""

result = pd.read_sql_query(query, conn)
result

,day_of_week,total_sales
0,None,187611.49
1,Tuesday,29125.44
2,Sunday,25486.26
3,Saturday,20890.62
4,Friday,17454.13
5,Wednesday,14454.94
6,Thursday,14122.46
7,Monday,13821.41
